# Step 1 Fixed-Gap Planner Evaluation

This notebook evaluates the trained causal Mimi → multipart-anchor planner on the **validation split only**. The test split remains untouched. It separates five questions that must not be conflated:

1. Does teacher-forced token prediction beat uniform, train-unigram, and previous-anchor-copy baselines?
2. How much does accuracy degrade when earlier anchors are generated rather than GT?
3. Does performance change when text or Mimi q0 is shuffled across clips?
4. Are errors concentrated in later rollout horizons or later RVQ levels?
5. How damaging are predicted anchor substitutions in codec latent/decoded space when the non-anchor gaps are held to GT?

The codec section is explicitly an **oracle-gap anchor-substitution diagnostic**. It is not Step 2 evaluation and must not be reported as end-to-end generated motion quality. Face is excluded.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer

def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'motion_generation').is_dir() and (candidate / 'SuSuInterActs').exists():
            return candidate
    raise RuntimeError('Could not locate the sentiAvatar-sandbox project root')

PROJECT_ROOT = find_project_root()
MOTION_DIR = PROJECT_ROOT / 'motion_generation'
if str(MOTION_DIR) not in sys.path:
    sys.path.insert(0, str(MOTION_DIR))

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

from models.step1_mimi_planner import (
    MimiQwenPlanner, Step1PlannerCollator, load_motion_tokens, load_text_map, read_split_names,
)
from scripts.export_multipart_motion_tokens import configure_strict_inference_math, load_part_codec
from scripts.train_step1_multipart_fixed_gap3 import (
    build_dataset, load_neutral_seed, resolve_data_paths, section,
)
from utils.step1_planner_evaluation import (
    ShuffledAudioDataset, codec_anchor_diagnostics, collect_fixed_gap_targets,
    evaluate_reference_baselines, evaluate_rollouts, greedy_rollout_item,
    teacher_forced_metrics, write_rollout_cache,
)

print('project:', PROJECT_ROOT)
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

## 1. Configuration

Teacher-forced evaluation and reference baselines default to the complete 635-clip validation split. Generated rollouts are more expensive, so start with 128 clips, then set `ROLLOUT_MAX_CLIPS = None` for the final validation report. Conditioning and codec diagnostics use smaller paired subsets by default.

In [ ]:
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' else torch.float32
CHECKPOINTS = {
    'best': PROJECT_ROOT / 'checkpoints/step1_multipart_fixed_gap3_main6000/best',
    'final': PROJECT_ROOT / 'checkpoints/step1_multipart_fixed_gap3_main6000/final',
}
TEACHER_FORCED_MAX_CLIPS = None
TEACHER_FORCED_BATCH_SIZE = 32
ROLLOUT_MAX_CLIPS = 128
CONDITION_MAX_CLIPS = 128
ROLLOUT_ABLATION_MAX_CLIPS = 32
DECODE_MAX_CLIPS = 32
NUM_WORKERS = 0
RUN_CONDITION_TESTS = True
RUN_ROLLOUT_ABLATIONS = True
RUN_CODEC_DIAGNOSTICS = True
OUTPUT_DIR = MOTION_DIR / 'outputs/step1_fixed_gap3_evaluation'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CODEC_CHECKPOINTS = {
    'upper': PROJECT_ROOT / 'checkpoints/causal_multipart_rvqvae/causal_rvq_upper_512x4_scratch/model/best.pth',
    'lower': PROJECT_ROOT / 'checkpoints/causal_multipart_rvqvae/causal_rvq_lower_512x4_scratch/model/best.pth',
    'feet': PROJECT_ROOT / 'checkpoints/causal_multipart_rvqvae/causal_rvq_feet_512x4_scratch/model/best.pth',
    'hands': PROJECT_ROOT / 'checkpoints/causal_multipart_rvqvae/causal_rvq_hands_512x4_scratch/model/best.pth',
}
math_mode = configure_strict_inference_math(DEVICE)
print('device:', DEVICE, '| dtype:', DTYPE)
print('strict math:', math_mode)

## 2. Recover the authoritative training contract

The checkpoint's `phase1_source_config.json`, not a subsequently edited YAML, defines the data split and seed policy used for this run. This also records whether the reported run was the 6,000-clip/50-epoch experiment.

In [ ]:
for label, path in CHECKPOINTS.items():
    if not path.is_dir():
        raise FileNotFoundError(f'{label} checkpoint missing: {path}')
source_config_path = CHECKPOINTS['final'] / 'phase1_source_config.json'
source_config = json.loads(source_config_path.read_text(encoding='utf-8'))
paths = resolve_data_paths(source_config)
data_config = section(source_config, 'data')
training_config = section(source_config, 'training')
train_names = read_split_names(paths['train_split'])
val_names_all = read_split_names(paths['eval_split'])
val_names = val_names_all[:TEACHER_FORCED_MAX_CLIPS] if TEACHER_FORCED_MAX_CLIPS else val_names_all
contract = {
    'train_clips': len(train_names),
    'validation_clips': len(val_names_all),
    'fixed_gap': data_config.get('fixed_gap'),
    'max_length': data_config.get('max_length'),
    'seed_mode': data_config.get('seed_mode'),
    'epochs': training_config.get('num_train_epochs'),
    'per_device_batch': training_config.get('per_device_train_batch_size'),
    'gradient_accumulation': training_config.get('gradient_accumulation_steps'),
}
display(pd.Series(contract, name='checkpoint training contract'))
assert len(train_names) == 6000, 'Expected the scientific 6,000-clip subset'
assert set(train_names).isdisjoint(val_names_all), 'Train/validation leakage detected'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINTS['final'], local_files_only=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
text_map = load_text_map(paths['text_json'])
neutral_seed = load_neutral_seed(data_config.get('neutral_seed_json'))
data_config = dict(data_config)
data_config['random_seed'] = int(training_config.get('seed', 42))
val_dataset = build_dataset(
    val_names, tokenizer=tokenizer, paths=paths, text_map=text_map,
    data_config=data_config, neutral_seed=neutral_seed, training=False,
)
collator = Step1PlannerCollator(tokenizer.pad_token_id, pad_to_multiple_of=8)
def make_loader(dataset, batch_size=TEACHER_FORCED_BATCH_SIZE):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == 'cuda', collate_fn=collator,
    )
def deterministic_subset(names, limit, seed=42):
    if limit is None or limit >= len(names):
        return list(names)
    generator = np.random.default_rng(seed)
    selected = np.sort(generator.choice(len(names), size=limit, replace=False))
    return [names[int(index)] for index in selected]
print('serialized validation examples:', len(val_dataset))
print('first item:', {k: val_dataset[0][k] for k in ('name', 'anchor_times')})

## 3. Reference baselines

Uniform chance (`1/512`) is not sufficient because RVQ usage can be imbalanced. We fit a smoothed per-slot unigram distribution on the exact 6,000 training clips and compare against copying the preceding GT anchor—the strongest simple persistence baseline under the current fixed schedule.

In [ ]:
train_targets, _, _ = collect_fixed_gap_targets(
    names=train_names, motion_token_dir=paths['motion_token_dir'],
    fixed_gap=int(data_config['fixed_gap']), require_causal=True,
)
val_targets, val_previous, val_target_index = collect_fixed_gap_targets(
    names=val_names, motion_token_dir=paths['motion_token_dir'],
    fixed_gap=int(data_config['fixed_gap']), require_causal=True,
)
baseline_summary, baseline_slots = evaluate_reference_baselines(
    train_targets=train_targets, validation_targets=val_targets,
    validation_previous=val_previous, alpha=1.0,
)
baseline_summary_df = pd.DataFrame(baseline_summary)
baseline_slots_df = pd.DataFrame(baseline_slots)
display(baseline_summary_df)
baseline_summary_df.to_csv(OUTPUT_DIR / 'baseline_summary.csv', index=False)
baseline_slots_df.to_csv(OUTPUT_DIR / 'baseline_per_slot.csv', index=False)

## 4. Full-validation teacher-forced comparison

Evaluate both `best/` and `final/`. The historical early-stopping `min_delta` allowed `final` to have a slightly lower raw validation CE than the recorded `best`, so neither directory should be assumed superior without recomputation.

In [ ]:
def load_planner(path: Path):
    try:
        loaded = MimiQwenPlanner.from_pretrained(path, dtype=DTYPE, local_files_only=True)
    except TypeError:
        loaded = MimiQwenPlanner.from_pretrained(path, torch_dtype=DTYPE, local_files_only=True)
    loaded.gradient_checkpointing_disable()
    loaded.language_model.config.use_cache = True
    return loaded.to(DEVICE).eval()

teacher_results = {}
teacher_summary_rows = []
teacher_slot_rows = []
for checkpoint_name, checkpoint_path in CHECKPOINTS.items():
    print('evaluating', checkpoint_name, checkpoint_path)
    planner = load_planner(checkpoint_path)
    result = teacher_forced_metrics(
        planner, make_loader(val_dataset), device=DEVICE, use_bf16=DEVICE.type == 'cuda',
    )
    teacher_results[checkpoint_name] = result
    teacher_summary_rows.append({'checkpoint': checkpoint_name, **result['summary']})
    teacher_slot_rows.extend({'checkpoint': checkpoint_name, **row} for row in result['slot_rows'])
    del planner
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
teacher_summary_df = pd.DataFrame(teacher_summary_rows).sort_values('cross_entropy')
teacher_slots_df = pd.DataFrame(teacher_slot_rows)
display(teacher_summary_df)
display(teacher_slots_df.pivot(index='slot_name', columns='checkpoint', values=['cross_entropy', 'accuracy', 'top5_accuracy']))
teacher_summary_df.to_csv(OUTPUT_DIR / 'teacher_forced_checkpoint_summary.csv', index=False)
teacher_slots_df.to_csv(OUTPUT_DIR / 'teacher_forced_per_slot.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_df = teacher_slots_df.copy()
for checkpoint_name, group in plot_df.groupby('checkpoint'):
    axes[0].plot(group['slot_name'], group['accuracy'] * 100, marker='o', label=checkpoint_name)
    axes[1].plot(group['slot_name'], group['cross_entropy'], marker='o', label=checkpoint_name)
axes[0].axhline(100 / 512, color='gray', linestyle='--', label='uniform chance')
axes[0].set_ylabel('Top-1 accuracy (%)')
axes[1].set_ylabel('Cross entropy')
for axis in axes:
    axis.tick_params(axis='x', rotation=65)
    axis.grid(alpha=0.25)
    axis.legend()
fig.suptitle('Teacher-forced validation by multipart RVQ slot')
fig.tight_layout()
plt.show()

## 5. Paired conditioning sensitivity

This is a sensitivity test, not a retrained ablation. Every target and motion prefix stays fixed while complete text or Mimi q0 is cyclically reassigned from another validation clip. A meaningful CE increase supports—but does not prove—that the planner uses that condition. A near-zero change is evidence that GT motion prefixes dominate.

In [ ]:
selected_checkpoint = str(teacher_summary_df.iloc[0]['checkpoint'])
selected_path = CHECKPOINTS[selected_checkpoint]
print('selected checkpoint:', selected_checkpoint, selected_path)
planner = load_planner(selected_path)
condition_summary_df = pd.DataFrame()
condition_datasets = {}
if RUN_CONDITION_TESTS:
    condition_names = deterministic_subset(val_names, CONDITION_MAX_CLIPS, seed=43)
    donor_names = condition_names[1:] + condition_names[:1]
    condition_base = build_dataset(
        condition_names, tokenizer=tokenizer, paths=paths, text_map=text_map,
        data_config=data_config, neutral_seed=neutral_seed, training=False,
    )
    shuffled_text_map = dict(text_map)
    for name, donor in zip(condition_names, donor_names):
        shuffled_text_map[name] = text_map[donor]
    shuffled_text = build_dataset(
        condition_names, tokenizer=tokenizer, paths=paths, text_map=shuffled_text_map,
        data_config=data_config, neutral_seed=neutral_seed, training=False,
    )
    shuffled_audio = ShuffledAudioDataset(
        condition_base, donor_names=donor_names, mimi_token_dir=paths['mimi_token_dir'],
    )
    condition_datasets = {'full': condition_base, 'shuffled_text': shuffled_text, 'shuffled_mimi_q0': shuffled_audio}
    condition_rows = []
    for condition, dataset in condition_datasets.items():
        measured = teacher_forced_metrics(planner, make_loader(dataset), device=DEVICE, use_bf16=DEVICE.type == 'cuda')
        condition_rows.append({'condition': condition, **measured['summary']})
    condition_summary_df = pd.DataFrame(condition_rows)
    reference_ce = float(condition_summary_df.loc[condition_summary_df.condition == 'full', 'cross_entropy'].iloc[0])
    reference_acc = float(condition_summary_df.loc[condition_summary_df.condition == 'full', 'accuracy'].iloc[0])
    condition_summary_df['delta_ce_vs_full'] = condition_summary_df['cross_entropy'] - reference_ce
    condition_summary_df['delta_accuracy_vs_full'] = condition_summary_df['accuracy'] - reference_acc
    display(condition_summary_df)
    condition_summary_df.to_csv(OUTPUT_DIR / 'condition_sensitivity_teacher_forced.csv', index=False)
else:
    print('condition tests disabled')

## 6. True generated-prefix rollout

Greedy decoding uses the known seed anchor, then generates every later 16-ID anchor autoregressively with a KV cache. Earlier generated anchors—not GT anchors—remain in the prefix. This measures both within-anchor and across-anchor exposure bias. The generated JSON files use the rollout-cache schema already accepted by the training dataset.

In [ ]:
rollout_names = deterministic_subset(val_names, ROLLOUT_MAX_CLIPS, seed=42)
rollout_dataset = build_dataset(
    rollout_names, tokenizer=tokenizer, paths=paths, text_map=text_map,
    data_config=data_config, neutral_seed=neutral_seed, training=False,
)
rollout_teacher = teacher_forced_metrics(
    planner, make_loader(rollout_dataset), device=DEVICE, use_bf16=DEVICE.type == 'cuda',
)
rollouts = []
for index in tqdm(range(len(rollout_dataset)), desc='greedy generated-prefix rollout'):
    rollouts.append(greedy_rollout_item(
        planner, rollout_dataset[index], device=DEVICE, use_bf16=DEVICE.type == 'cuda',
    ))
rollout_metrics = evaluate_rollouts(rollouts)
rollout_targets, rollout_previous, _ = collect_fixed_gap_targets(
    names=rollout_names, motion_token_dir=paths['motion_token_dir'],
    fixed_gap=int(data_config['fixed_gap']), require_causal=True,
)
assert np.array_equal(rollout_metrics['labels'], rollout_targets), 'Rollout/baseline target order differs'
rollout_baseline_summary, _ = evaluate_reference_baselines(
    train_targets=train_targets, validation_targets=rollout_targets,
    validation_previous=rollout_previous, alpha=1.0,
)
rollout_baseline_df = pd.DataFrame(rollout_baseline_summary)
rollout_unigram_accuracy = float(rollout_baseline_df.loc[rollout_baseline_df.baseline == 'train_unigram_majority', 'accuracy'].iloc[0])
rollout_copy_accuracy = float(rollout_baseline_df.loc[rollout_baseline_df.baseline == 'previous_gt_anchor_copy', 'accuracy'].iloc[0])
rollout_summary = {
    **rollout_metrics['summary'],
    'teacher_forced_accuracy_same_subset': rollout_teacher['summary']['accuracy'],
    'teacher_forced_ce_same_subset': rollout_teacher['summary']['cross_entropy'],
    'unigram_accuracy_same_subset': rollout_unigram_accuracy,
    'previous_copy_accuracy_same_subset': rollout_copy_accuracy,
}
rollout_summary['accuracy_drop_from_teacher_forcing'] = rollout_summary['accuracy'] - rollout_summary['teacher_forced_accuracy_same_subset']
rollout_summary['accuracy_margin_over_previous_copy'] = rollout_summary['accuracy'] - rollout_copy_accuracy
rollout_summary_df = pd.DataFrame([rollout_summary])
rollout_slots_df = pd.DataFrame(rollout_metrics['slot_rows'])
rollout_horizon_df = pd.DataFrame(rollout_metrics['horizon_rows'])
display(rollout_summary_df)
display(rollout_slots_df)
rollout_summary_df.to_csv(OUTPUT_DIR / 'generated_prefix_rollout_summary.csv', index=False)
rollout_slots_df.to_csv(OUTPUT_DIR / 'generated_prefix_rollout_per_slot.csv', index=False)
rollout_horizon_df.to_csv(OUTPUT_DIR / 'generated_prefix_rollout_horizon.csv', index=False)
rollout_cache_dir = OUTPUT_DIR / 'validation_rollout_cache'
write_rollout_cache(rollouts, rollout_cache_dir)
print('validation rollout cache:', rollout_cache_dir)

In [ ]:
rollout_horizon_df['horizon_bin'] = pd.cut(
    rollout_horizon_df['relative_horizon'], bins=[-1e-9, .25, .5, .75, 1.0],
    labels=['0–25%', '25–50%', '50–75%', '75–100%'], include_lowest=True,
)
horizon_summary = rollout_horizon_df.groupby('horizon_bin', observed=True)[['accuracy', 'q0_accuracy', 'confidence', 'entropy']].mean().reset_index()
display(horizon_summary)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(horizon_summary['horizon_bin'].astype(str), horizon_summary['accuracy'] * 100, marker='o', label='all slots')
axes[0].plot(horizon_summary['horizon_bin'].astype(str), horizon_summary['q0_accuracy'] * 100, marker='o', label='q0')
axes[0].set_ylabel('Generated-prefix accuracy (%)')
axes[0].legend()
axes[1].plot(horizon_summary['horizon_bin'].astype(str), horizon_summary['confidence'], marker='o', label='confidence')
axes[1].set_ylabel('Mean predicted confidence')
for axis in axes:
    axis.set_xlabel('Relative utterance horizon')
    axis.grid(alpha=.25)
fig.tight_layout()
plt.show()

## 7. Generated-prefix conditioning sensitivity

Repeat greedy rollout on a small paired subset with shuffled text and shuffled q0. This is more diagnostic than teacher-forced sensitivity because the model can no longer hide conditioning failures behind GT motion history.

In [ ]:
rollout_condition_df = pd.DataFrame()
if RUN_ROLLOUT_ABLATIONS and RUN_CONDITION_TESTS:
    count = min(ROLLOUT_ABLATION_MAX_CLIPS, len(condition_datasets['full']))
    condition_rollout_rows = []
    for condition, dataset in condition_datasets.items():
        condition_results = [
            greedy_rollout_item(planner, dataset[index], device=DEVICE, use_bf16=DEVICE.type == 'cuda')
            for index in tqdm(range(count), desc=f'rollout {condition}')
        ]
        measured = evaluate_rollouts(condition_results)['summary']
        condition_rollout_rows.append({'condition': condition, **measured})
    rollout_condition_df = pd.DataFrame(condition_rollout_rows)
    full_accuracy = float(rollout_condition_df.loc[rollout_condition_df.condition == 'full', 'accuracy'].iloc[0])
    rollout_condition_df['delta_accuracy_vs_full'] = rollout_condition_df['accuracy'] - full_accuracy
    display(rollout_condition_df)
    rollout_condition_df.to_csv(OUTPUT_DIR / 'condition_sensitivity_generated_prefix.csv', index=False)
else:
    print('generated-prefix condition tests disabled')

## 8. Codec-space anchor diagnostics

Replace only the scheduled GT anchor IDs with generated IDs while retaining every non-anchor GT token. Compare against replacing those anchors with the previous GT anchor. This measures anchor semantic damage under an oracle gap and reports latent distance, rotation error, velocity error, and lower-body root drift. It does **not** include Step 2.

In [ ]:
codec_rows_df = pd.DataFrame()
if RUN_CODEC_DIAGNOSTICS:
    missing = {part: str(path) for part, path in CODEC_CHECKPOINTS.items() if not path.is_file()}
    if missing:
        raise FileNotFoundError(f'Codec checkpoints missing: {missing}')
    del planner
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    codecs = {part: load_part_codec(path, DEVICE) for part, path in CODEC_CHECKPOINTS.items()}
    diagnostic_rows = []
    for result in tqdm(rollouts[:DECODE_MAX_CLIPS], desc='codec anchor diagnostics'):
        motion_path = paths['motion_token_dir'].joinpath(*Path(result.name).parts).with_suffix('.json')
        dense_tokens, _ = load_motion_tokens(motion_path, require_causal=True)
        rows = codec_anchor_diagnostics(
            codecs=codecs, dense_tokens=dense_tokens, anchor_times=result.anchor_times,
            predicted_anchors=result.predicted_anchors, device=DEVICE,
        )
        diagnostic_rows.extend({'name': result.name, **row} for row in rows)
    codec_rows_df = pd.DataFrame(diagnostic_rows)
    codec_summary = codec_rows_df.groupby(['variant', 'part'], as_index=False).mean(numeric_only=True)
    display(codec_summary)
    codec_rows_df.to_csv(OUTPUT_DIR / 'codec_anchor_diagnostics_per_clip.csv', index=False)
    codec_summary.to_csv(OUTPUT_DIR / 'codec_anchor_diagnostics_summary.csv', index=False)
else:
    print('codec diagnostics disabled')

## 9. Report and decision checklist

This report deliberately avoids a single pass/fail score. A usable fixed-gap baseline should beat unigram and previous-anchor copying, retain useful q0 accuracy under generated prefixes, show limited horizon collapse, react adversely to mismatched conditions, and improve codec-space anchor substitutions over copying. Frozen Step 2 evaluation remains the next independent gate.

In [ ]:
def records(frame: pd.DataFrame):
    return json.loads(frame.to_json(orient='records')) if not frame.empty else []

unigram_accuracy = float(baseline_summary_df.loc[baseline_summary_df.baseline == 'train_unigram_majority', 'accuracy'].iloc[0])
copy_accuracy = float(baseline_summary_df.loc[baseline_summary_df.baseline == 'previous_gt_anchor_copy', 'accuracy'].iloc[0])
selected_teacher = teacher_results[selected_checkpoint]['summary']
decision = {
    'teacher_forced_beats_unigram_accuracy': bool(selected_teacher['accuracy'] > unigram_accuracy),
    'teacher_forced_beats_previous_copy_accuracy': bool(selected_teacher['accuracy'] > copy_accuracy),
    'generated_prefix_beats_previous_copy_same_subset': bool(rollout_summary['accuracy'] > rollout_summary['previous_copy_accuracy_same_subset']),
    'generated_prefix_q0_accuracy': float(rollout_summary['q0_accuracy']),
    'generated_prefix_accuracy_drop': float(rollout_summary['accuracy_drop_from_teacher_forcing']),
    'test_split_used': False,
    'step2_evaluated': False,
}
report = {
    'contract': contract,
    'selected_checkpoint': selected_checkpoint,
    'math_mode': math_mode,
    'baseline_summary': records(baseline_summary_df),
    'teacher_forced_summary': records(teacher_summary_df),
    'condition_sensitivity_teacher_forced': records(condition_summary_df),
    'generated_prefix_rollout': rollout_summary,
    'condition_sensitivity_generated_prefix': records(rollout_condition_df),
    'decision': decision,
}
report_path = OUTPUT_DIR / 'evaluation_report.json'
report_path.write_text(json.dumps(report, indent=2, allow_nan=True), encoding='utf-8')
display(pd.Series(decision, name='decision diagnostics'))
print('report:', report_path)
print('outputs:', OUTPUT_DIR)

### Interpretation discipline

- Teacher-forced accuracy alone does not establish reliability because the model sees GT prior anchors and earlier GT slots.
- Shuffling sensitivity is evidence of condition use, not proof of causal semantics or correct attention.
- q1–q3 exact IDs may be difficult while decoded errors remain modest; report both token and codec-space results.
- Oracle-gap decoding isolates anchor substitutions. Do not call it generated motion or Step 2 quality.
- Do not inspect the test split until the architecture, decoding policy, and evaluation gates are frozen.
- After this notebook, run the selected generated anchors through the frozen Step 2 reference and compare GT-anchor, predicted-anchor, copy-anchor, and no-anchor conditions.